# Reinforcement Learning for LLMs


Lecture 13 ended with a pretrained model that had been fine-tuned on
demonstrations. This lecture is about the third stage of the LLM lifecycle: the
one where the model generates its own responses, something scores them, and the
parameters move toward responses that score well. That stage is reinforcement
learning, and since early 2025 it is where most of the capability gains in
reasoning and agentic models have come from. The lecture is self-contained: it
opens with the fifteen minutes of RL that the rest needs (the MDP, the policy
gradient theorem, REINFORCE and baselines, following UDL chapter 19), then goes
through the methods in the order the field adopted them: PPO with a learned
reward model (RLHF), DPO, and GRPO with verifiable rewards, ending with the
2025–26 fixes to GRPO, agentic multi-turn RL, and what you can run on one GPU.

Notation. UDL chapter 19 writes a stochastic policy as
$\pi[a_t \mid s_t, \boldsymbol{\theta}]$, a trajectory as $\tau$ and its return
as $r[\tau]$; the LLM papers write $\pi_{\theta}(y \mid x)$ for the same thing.
Section 2 uses UDL's form. From PPO onward every equation is quoted from a
primary paper, so the lecture switches to the papers' form and says so at the
switch. The quizzable derivations are the policy-gradient theorem (section 2)
and the closed-form optimal policy behind DPO (section 4).


## Today's roadmap

1. Why RL after pretraining: what SFT cannot do; a language model as a policy
2. RL in 15 minutes: MDP, return, value, advantage; the policy-gradient theorem; REINFORCE; baselines
3. From REINFORCE to PPO; reward models and RLHF; reward hacking
4. DPO: the closed-form optimal policy and the loss it implies
5. GRPO and verifiable rewards: DeepSeek-R1-Zero, and the 2025–26 fixes
6. Agentic RL: multi-turn rollouts with tools
7. Practicalities on one GPU
8. Summary


# Why RL after pretraining

Lecture 13 built an instruct model with supervised fine-tuning. This section
asks what that model still cannot learn from demonstrations alone, and sets up
the vocabulary that lets us treat a language model as something RL can train.


## What SFT can and cannot do

SFT minimizes next-token cross-entropy on $(x, y)$ pairs written by people. That is
maximum likelihood on *demonstrations*.

- **It can** teach a format, a persona, a tool-calling syntax, and the skills
  visible in the demonstrations. A few thousand good examples go a long way
  (LIMA, lecture 13).
- **It cannot express "this answer is worse than that one."** Every target
  token gets the same treatment; there is no negative signal.
- **It cannot use a reward computed by a program.** "The unit tests passed" is
  not a token sequence you can take the likelihood of.
- **It never sees the model's own mistakes.** Training is on human text; at
  inference the model conditions on its own earlier tokens. Errors compound,
  and SFT has no mechanism to correct them.
- **It is capped by the demonstrator.** The DeepSeek-R1 report argues that SFT
  on human-written reasoning can even *impede* exploration, because human
  solutions omit the reflection and verification steps the model needs to
  discover ([DeepSeek-AI 2025](https://arxiv.org/abs/2501.12948), supplementary
  section 1).


RL fixes all four: the model samples its **own** responses, a **scalar reward**
scores them, and the gradient pushes probability toward the better ones and
away from the worse ones.


## The LLM lifecycle

![The three training stages of a 2026 language model. Lecture 13 covered stages 1 and 2; this lecture is stage 3. Source: instructor figure.](figs/14-RL-for-LLMs/llm-lifecycle.png)

Stage 3 is the only stage whose training data is generated by the model being
trained. That single fact explains most of what follows: the cost (you must
sample), the instability (the data distribution moves as you train), and the
need for a reference model (to keep the moving distribution from going
somewhere strange).


## A language model is a policy

| RL term | For a language model |
|---|---|
| state $s_t$ | the prompt plus the tokens generated so far, $(x, y_{<t})$ |
| action $a_t$ | the next token $y_t$, one of $V$ choices |
| policy $\pi[a_t \mid s_t, \boldsymbol{\theta}]$ | the softmax over the vocabulary, exactly the network's output |
| transition | deterministic: append the token |
| trajectory / episode $\tau$ | one complete response $y = (y_1, \ldots, y_T)$ |
| reward | usually **one number at the end**, $r(x, y)$; zero at every other step |
| return $r[\tau]$ | that same number (no discounting within a response) |

Two things are unusual compared with a game or a robot: the transition is known
and trivial, and the reward is sparse and terminal. Both simplify the
mathematics. The thing that is *harder* is the action space: $V \approx 150{,}000$
choices per step, for thousands of steps.


# RL in 15 minutes

Half of you saw MDPs and policy gradients conceptually in DS340; the other half
have not. This section is self-contained and follows UDL chapter 19, sections
19.1 and 19.5. If you want more, UDL's chapter 19 is a careful 30 pages; we
need only its policy-gradient third.


## The Markov decision process

An MDP is a tuple of states $s$, actions $a$, a transition distribution
$\mathrm{Pr}(s_{t+1} \mid s_t, a_t)$, a reward $r_{t+1} = \mathrm{r}[s_t, a_t]$, and
a discount $\gamma \in (0, 1]$. An agent follows a **policy**, a distribution
over actions given the state:

$$
a_t \sim \pi[a_t \mid s_t, \boldsymbol{\theta}].
$$

Running the policy from an initial state produces a **trajectory**
$\tau = [s_1, a_1, s_2, a_2, \ldots, s_T, a_T]$ with probability

$$
\mathrm{Pr}(\tau \mid \boldsymbol{\theta}) = \mathrm{Pr}(s_1) \prod_{t=1}^{T} \pi[a_t \mid s_t, \boldsymbol{\theta}]\, \mathrm{Pr}(s_{t+1} \mid s_t, a_t).
$$

The **Markov** property is that the next state depends only on the current state
and action. For an LLM the state is the whole prefix, so it holds trivially.


## Return, value and advantage

The **return** from time $t$ is the discounted sum of future rewards,
$G_t = \sum_{k=0}^{T-t} \gamma^k r_{t+k+1}$. The return of a whole trajectory
is $r[\tau] = G_1$.

The **state value** and **action value** are expected returns under the policy:

$$
\mathrm{v}[s_t \mid \pi] = \mathbb{E}\big[G_t \mid s_t\big], \qquad
\mathrm{q}[s_t, a_t \mid \pi] = \mathbb{E}\big[G_t \mid s_t, a_t\big].
$$

The **advantage** of an action is how much better it was than the policy's
average from that state:

$$
\mathrm{A}[s_t, a_t] = \mathrm{q}[s_t, a_t \mid \pi] - \mathrm{v}[s_t \mid \pi].
$$

Every algorithm in this lecture is a different way of *estimating* the
advantage of the tokens a model produced, and then pushing the log-probability of
those tokens in proportion to it.



## The objective, and why we cannot just differentiate it

We want parameters that maximize the expected return:

$$
J[\boldsymbol{\theta}] = \mathbb{E}_{\tau \sim \mathrm{Pr}(\tau \mid \boldsymbol{\theta})}\big[r[\tau]\big] = \int \mathrm{Pr}(\tau \mid \boldsymbol{\theta})\, r[\tau]\, d\tau.
$$

- In supervised learning the data distribution is fixed and the loss depends on
  $\boldsymbol{\theta}$. Here it is reversed: the **reward does not depend on
  $\boldsymbol{\theta}$ at all**; the *distribution of trajectories* does.
- The reward may not be differentiable (a test suite is not), and even if it
  were, sampling a token is a discrete choice with no gradient through it.
- So we need the gradient of an expectation with respect to the distribution's
  parameters, estimated from samples. The trick for that is one line.


## The log-derivative trick

For any density $\mathrm{Pr}(\tau \mid \boldsymbol{\theta})$:

$$
\frac{\partial \mathrm{Pr}(\tau \mid \boldsymbol{\theta})}{\partial \boldsymbol{\theta}}
= \mathrm{Pr}(\tau \mid \boldsymbol{\theta}) \,
\frac{\partial \log \mathrm{Pr}(\tau \mid \boldsymbol{\theta})}{\partial \boldsymbol{\theta}},
$$

because $\partial \log f / \partial z = (1/f)\, \partial f / \partial z$ (UDL eq. 19.26).

It turns a derivative of a density into the density times a derivative of a
log-density. The first factor lets us sample; the second is something a network
can compute by backpropagation, because $\log \pi$ is just the log-softmax
output.


## The policy-gradient theorem: the derivation

Differentiate the objective, apply the trick, and recognize an expectation:

$$
\begin{aligned}
\frac{\partial J}{\partial \boldsymbol{\theta}}
&= \int \frac{\partial \mathrm{Pr}(\tau \mid \boldsymbol{\theta})}{\partial \boldsymbol{\theta}}\, r[\tau]\, d\tau
 = \int \mathrm{Pr}(\tau \mid \boldsymbol{\theta})\, \frac{\partial \log \mathrm{Pr}(\tau \mid \boldsymbol{\theta})}{\partial \boldsymbol{\theta}}\, r[\tau]\, d\tau
 = \mathbb{E}_{\tau}\!\left[ \frac{\partial \log \mathrm{Pr}(\tau \mid \boldsymbol{\theta})}{\partial \boldsymbol{\theta}}\, r[\tau] \right].
\end{aligned}
$$

Now expand the log of the trajectory probability. Only the policy factors depend on $\boldsymbol{\theta}$:

$$
\log \mathrm{Pr}(\tau \mid \boldsymbol{\theta}) = \log \mathrm{Pr}(s_1) + \sum_{t=1}^{T} \log \pi[a_t \mid s_t, \boldsymbol{\theta}] + \sum_{t=1}^{T} \log \mathrm{Pr}(s_{t+1} \mid s_t, a_t),
$$

so the initial-state and transition terms vanish under $\partial/\partial\boldsymbol{\theta}$, leaving

$$
\boxed{\;\frac{\partial J}{\partial \boldsymbol{\theta}} = \mathbb{E}_{\tau \sim \mathrm{Pr}(\tau \mid \boldsymbol{\theta})}\!\left[ \sum_{t=1}^{T} \frac{\partial \log \pi[a_t \mid s_t, \boldsymbol{\theta}]}{\partial \boldsymbol{\theta}}\; r[\tau] \right]\;}
\qquad \text{(after UDL eqs. 19.27–19.29)}
$$

Three things to notice. (1) **No model of the environment is needed**: the
transition terms dropped out. (2) The gradient is an expectation under the
current policy, so it can be estimated by **sampling trajectories from the
current policy** and averaging. (3) The estimate is unbiased but, as we will see,
has very high variance.



## Two refinements that do not change the expectation

**Reward to go.** An action at time $t$ cannot affect rewards received before
$t$. Replacing the full return $r[\tau]$ by the return *from $t$ on* leaves the
expectation unchanged and lowers the variance (UDL eq. 19.31):

$$
\frac{\partial J}{\partial \boldsymbol{\theta}} = \mathbb{E}_{\tau}\!\left[ \sum_{t=1}^{T} \frac{\partial \log \pi[a_t \mid s_t, \boldsymbol{\theta}]}{\partial \boldsymbol{\theta}}\; \sum_{k \geq t} r_{k+1} \right].
$$

**For an LLM with one terminal reward** the reward-to-go from every token is the
same number $r(x, y)$, so the two forms coincide, and the sum of per-token
log-probabilities is the log-probability of the whole response:

$$
\frac{\partial J}{\partial \boldsymbol{\theta}} = \mathbb{E}_{y \sim \pi}\!\left[ r(x, y)\, \frac{\partial \log \pi(y \mid x, \boldsymbol{\theta})}{\partial \boldsymbol{\theta}} \right],
\qquad
\log \pi(y \mid x, \boldsymbol{\theta}) = \sum_{t=1}^{T} \log \pi[y_t \mid x, y_{<t}, \boldsymbol{\theta}].
$$

Every token in a good response is pushed up by the same amount; every token in a
bad one is pushed down. The credit-assignment problem *within* a response is
left entirely to the model.


## The picture

![The policy-gradient update in one dimension. Samples are drawn from the current policy; the arrows push log-probability up for samples with positive advantage and down for negative advantage; the dotted curve is the policy after the step. Source: instructor figure.](figs/14-RL-for-LLMs/policy-gradient-picture.png)

UDL's figure 19.15 makes the same point with five trajectories: the biggest
updates come from trajectories that are *unusual* under the current policy but
score well. Common, well-scoring trajectories need no change, which is what the
$1/\mathrm{Pr}(\tau \mid \boldsymbol{\theta})$ factor in the derivation does.


## REINFORCE: the algorithm

[Williams (1992)](https://link.springer.com/article/10.1007/BF00992696) turned the
theorem into a Monte Carlo method. Repeat:

1. Sample $I$ trajectories $\tau^{(i)}$ from the current policy $\pi[\cdot \mid \cdot, \boldsymbol{\theta}]$.
2. Compute the reward-to-go $r[\tau^{(i)}_t]$ for each step (for an LLM, the single terminal reward).
3. Take a gradient **ascent** step on the sampled estimate (UDL eq. 19.34;
   19.33 is the per-sample update):
$$
\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} + \alpha \cdot \frac{1}{I} \sum_{i=1}^{I} \sum_{t=1}^{T} \frac{\partial \log \pi[a^{(i)}_t \mid s^{(i)}_t, \boldsymbol{\theta}]}{\partial \boldsymbol{\theta}}\; r[\tau^{(i)}_t]
\qquad \text{(UDL eq. 19.34)}.
$$
4. **Throw the samples away.** They came from the old $\boldsymbol{\theta}$; the theorem needs samples from the current one.


In code this is the cross-entropy loss you already have, **weighted per
sequence by its return**: `loss = -(logprobs.sum(-1) * reward).mean()`. SFT is
the special case where the "samples" are human and every weight is 1.


## Variance reduction: subtract a baseline

Suppose every trajectory scores between 90 and 100. REINFORCE pushes *all* of
them up, nearly equally, and the useful signal (which ones scored 100) is a
small difference between large numbers. Subtract a **baseline** $b$ that does
not depend on the action:

$$
\frac{\partial J}{\partial \boldsymbol{\theta}} = \mathbb{E}_{\tau}\!\left[ \sum_{t} \frac{\partial \log \pi[a_t \mid s_t, \boldsymbol{\theta}]}{\partial \boldsymbol{\theta}}\; \big(r[\tau_t] - b\big) \right].
$$

**Why this is still unbiased** (UDL eq. 19.35). The extra term has expectation zero,
because for any distribution the expected score is zero:

$$
\mathbb{E}_{a \sim \pi}\!\left[ \frac{\partial \log \pi[a \mid s, \boldsymbol{\theta}]}{\partial \boldsymbol{\theta}} \right]
= \sum_{a} \pi[a \mid s, \boldsymbol{\theta}] \frac{\partial \log \pi[a \mid s, \boldsymbol{\theta}]}{\partial \boldsymbol{\theta}}
= \sum_{a} \frac{\partial \pi[a \mid s, \boldsymbol{\theta}]}{\partial \boldsymbol{\theta}}
= \frac{\partial}{\partial \boldsymbol{\theta}} \sum_{a} \pi[a \mid s, \boldsymbol{\theta}]
= \frac{\partial 1}{\partial \boldsymbol{\theta}} = 0.
$$


So $b$ can be anything that does not depend on the sampled action: a running
mean of returns, a **learned value function** $\mathrm{v}[s_t]$ (then
$r[\tau_t] - \mathrm{v}[s_t]$ estimates the advantage; this is *actor-critic*, and
it is what PPO does), or the **mean reward of other samples for the same
prompt** (this is what GRPO does). The choice of baseline is the main
difference between the LLM algorithms.


Both claims on a three-armed bandit with rewards near 90, 95 and 100 and a
softmax policy: the one-sample estimates average to the exact gradient with or
without the baseline, the baseline cuts their variance by a factor of hundreds,
and over 20 seeds of 200 REINFORCE steps only the baseline runs reliably find
the best arm.

In [ ]:
import numpy as np

mu = np.array([90.0, 95.0, 100.0])   # mean reward of each arm
sigma = 2.0                          # reward noise

def policy(theta):                   # softmax over three logits: the whole "network"
    e = np.exp(theta - theta.max())
    return e / e.sum()

def score(a, pi):                    # d log pi[a] / d theta = onehot(a) - pi
    return np.eye(len(pi))[a] - pi

rng = np.random.default_rng(0)
pi = policy(np.zeros(3))
exact = pi * (mu - pi @ mu)          # dJ/dtheta_k = pi_k (mu_k - E[r])
n = 200_000
a = rng.choice(3, size=n, p=pi)
r = mu[a] + sigma * rng.standard_normal(n)
g_plain = score(a, pi) * r[:, None]                 # one-sample REINFORCE estimates
g_base = score(a, pi) * (r - pi @ mu)[:, None]      # same, with baseline b = E[r]
print("exact gradient     ", np.round(exact, 3))
for name, g in [("no baseline", g_plain), ("with baseline", g_base)]:
    se = np.sqrt(g.var(0) / n)
    print(f"{name:<19} mean {np.round(g.mean(0), 3)} +/- {se.max():.3f}, "
          f"variance per sample {g.var(0).sum():7.1f}")

def reinforce(use_baseline, seed, steps=200, batch=8, lr=0.01):
    rng = np.random.default_rng(seed)
    theta, b = np.zeros(3), 0.0
    for step in range(steps):
        pi = policy(theta)
        a = rng.choice(3, size=batch, p=pi)
        r = mu[a] + sigma * rng.standard_normal(batch)
        weight = r - b if use_baseline else r
        theta += lr * (score(a, pi) * weight[:, None]).mean(0)    # gradient ascent
        b = r.mean() if step == 0 else 0.9 * b + 0.1 * r.mean()   # running mean of past returns
    return policy(theta)[2]

print("\nP(best arm) after 200 steps, 20 seeds:")
for name, use_b in [("no baseline", False), ("with baseline", True)]:
    p_best = np.array([reinforce(use_b, seed) for seed in range(20)])
    print(f"{name:<19} mean {p_best.mean():.3f}, std {p_best.std():.3f}, worst run {p_best.min():.3f}")

## On-policy sampling is the cost

The theorem's expectation is under the **current** policy. Each gradient step
therefore needs fresh samples, and for an LLM a sample is a full generation:

- Thousands of sequential decode steps per response, each memory-bound
  (lecture 12's roofline; lecture 15 will quantify).
- Typically 8 to 64 responses per prompt, hundreds of prompts per step.
- Generation commonly takes longer than the forward-backward pass of the step
  it feeds; production systems run it on separate GPUs with an inference
  engine (vLLM, SGLang) and ship weights across after each update.


Two ways out, and both appear next: **reuse each batch of samples for several
gradient steps** while correcting for the fact that the policy has moved (PPO,
GRPO), or **avoid sampling altogether** by training on a fixed dataset of
comparisons (DPO).


# From REINFORCE to PPO, and where the reward comes from

The policy gradient tells us how to move the parameters given a reward. This
section makes the update safe enough to take several steps per batch of samples
(PPO), and then asks where a scalar reward for an open-ended response comes
from in the first place (a reward model trained on human comparisons). From
here on the notation is the one the papers use: $\pi_{\theta}(y \mid x)$ for the
policy, $\pi_{\text{ref}}$ for the frozen reference model, $\pi_{\theta_{\text{old}}}$
for the policy that generated the current batch.


## Reusing samples: the importance ratio

Samples came from $\pi_{\theta_{\text{old}}}$ but we want the gradient under
$\pi_{\theta}$. Importance sampling corrects the mismatch. Per token, define

$$
\rho_t(\theta) = \frac{\pi_{\theta}(y_t \mid x, y_{<t})}{\pi_{\theta_{\text{old}}}(y_t \mid x, y_{<t})},
\qquad
L^{\text{IS}}(\theta) = \mathbb{E}_{y \sim \pi_{\theta_{\text{old}}}}\!\left[ \sum_t \rho_t(\theta)\, \hat A_t \right].
$$

At $\theta = \theta_{\text{old}}$ the ratio is 1 and the gradient of $L^{\text{IS}}$
is exactly the policy gradient (check: $\partial \rho_t / \partial \theta = \rho_t\, \partial \log \pi_{\theta} / \partial \theta$).

The problem: nothing stops the optimizer from driving $\rho_t$ to 10 for a token
with $\hat A_t > 0$. The estimate is only trustworthy near $\theta_{\text{old}}$,
and a large ratio means a large policy change based on data that no longer
represents the policy. TRPO constrained the KL between old and new policies;
PPO replaced the constraint with a clip.


## PPO: the clipped objective

[Schulman et al. 2017](https://arxiv.org/abs/1707.06347), eq. 7:

$$
L^{\text{CLIP}}(\theta) = \mathbb{E}_t\!\left[ \min\!\Big( \rho_t(\theta)\, \hat A_t,\;\; \mathrm{clip}\big(\rho_t(\theta),\, 1-\epsilon,\, 1+\epsilon\big)\, \hat A_t \Big) \right],
\qquad \epsilon = 0.2 \text{ in the paper.}
$$

![The clipped objective as a function of the ratio. For positive advantage the objective is flat above $1+\epsilon$; for negative advantage it is flat below $1-\epsilon$. The $\min$ keeps the unclipped branch wherever it is more pessimistic. Redrawn after figure 1 of Schulman et al. 2017. Source: instructor figure.](figs/14-RL-for-LLMs/ppo-clipping.png)

Read the figure as "the gradient is zero once the ratio has moved far enough in
the direction the advantage wanted." That is the whole trick: the objective
cannot be improved by moving the ratio beyond the clip range, so a few epochs of
SGD on one batch of samples stay in a trust region without ever computing a KL.
The $\min$ matters: it makes the bound one-sided, so a ratio that moved the
*wrong* way is always corrected.

The per-token objective and its derivative with respect to the ratio. The
derivative is zero exactly where the figure is flat, so a token whose ratio
has left the trust region in the advantage's direction stops receiving
gradient; one that moved the wrong way never does.

In [ ]:
import numpy as np

def ppo_clip(ratio, adv, eps_low=0.2, eps_high=0.2):
    """Per-token L^CLIP and its derivative with respect to the ratio."""
    clipped = np.clip(ratio, 1 - eps_low, 1 + eps_high)
    obj = np.minimum(ratio * adv, clipped * adv)
    grad = np.where(ratio * adv <= clipped * adv, adv, 0.0)   # the min kept the unclipped branch
    return obj, grad

print(f"{'ratio':>6} {'A':>4} {'L_CLIP':>7} {'dL/dratio':>10}")
for adv in (+1.0, -1.0):
    for ratio in (0.7, 0.9, 1.0, 1.1, 1.3):
        obj, grad = ppo_clip(ratio, adv)
        print(f"{ratio:>6.1f} {adv:>+4.0f} {obj:>7.2f} {grad:>+10.0f}")

## PPO for a language model in one slide

The recipe used by [Ziegler et al. 2019](https://arxiv.org/abs/1909.08593),
[Stiennon et al. 2020](https://arxiv.org/abs/2009.01325) and InstructGPT:

**Four networks.** Policy $\pi_{\theta}$ (trained), reference $\pi_{\text{ref}}$
(frozen copy of the SFT model), reward model $r_{\psi}(x, y)$ (frozen), value
network $V_{\phi}(s_t)$ (trained; usually a scalar head on a copy of the policy).

**Per-token reward.** The reward model scores the finished response; a KL
penalty to the reference is spread over tokens:

$$
r_t = -\beta \log \frac{\pi_{\theta}(y_t \mid x, y_{<t})}{\pi_{\text{ref}}(y_t \mid x, y_{<t})} + \mathbb{1}[t = T]\, r_{\psi}(x, y).
$$

**Advantage** $\hat A_t$ by generalized advantage estimation
([Schulman et al. 2015](https://arxiv.org/abs/1506.02438)) from the per-token
rewards and the value network's predictions; $\lambda$ trades its bias against
its variance.

**Loss**, minimized for a few epochs per batch: $-L^{\text{CLIP}}(\theta)$,
plus a squared error that trains $V_{\phi}(s_t)$ toward $\hat G_t$, with
$\hat G_t$ the empirical return (reward-to-go), minus an entropy bonus
(PPO paper, eq. 9).

**One step:** sample $N$ responses from $\pi_{\theta_{\text{old}}}$; score with
$r_{\psi}$; compute $r_t$, $V_{\phi}$, $\hat A_t$; run $K$ epochs of minibatch
SGD on the loss; set $\theta_{\text{old}} \leftarrow \theta$.

**What hurts.** Four model copies in memory; the value network is as large as
the policy and must predict the *final* reward from a *partial* response, which
is hard for long chains of thought (the DeepSeek-R1 report's argument for
dropping it, section 5); several sensitive hyperparameters ($\lambda$, $c_1$,
$\beta$, $\epsilon$, $K$).


## Where does the reward come from? Reward models

Lecture 13 defined the **Bradley–Terry reward model**: an LLM with a scalar head
$r_{\psi}(x, y)$, fit by maximum likelihood on human comparisons through
$\mathrm{Pr}(y_w \succ y_l \mid x) = \sigma\big( r_{\psi}(x, y_w) - r_{\psi}(x, y_l) \big)$.

The fact to carry: the loss is logistic regression on the *difference* of two
scores for the same prompt, so $r_{\psi}$ is defined only up to a per-prompt
additive constant. DPO (section 4) needs exactly that.


## The RLHF pipeline: InstructGPT

![The three steps of RLHF as run for InstructGPT: supervised fine-tuning on demonstrations, reward-model training on labeler rankings, and PPO against the reward model. Figure 2 of [Ouyang et al. 2022](https://arxiv.org/abs/2203.02155).](figs/14-RL-for-LLMs/instructgpt-pipeline.png)

Labelers ranked $K = 4$ to $9$ responses per prompt, which yields
$\binom{K}{2}$ comparisons each, and the reward-model loss averages over all of
them. The paper's headline result is that labelers preferred the 1.3B-parameter
InstructGPT to the 175B-parameter GPT-3: a hundred times fewer parameters,
trained on tens of thousands of comparisons, beat scale on "do what the user
asked."


## The RLHF objective

InstructGPT's RL objective ([Ouyang et al. 2022](https://arxiv.org/abs/2203.02155),
eq. 2, in our notation):

$$
\max_{\theta}\;\; \mathbb{E}_{x \sim \mathcal{D},\; y \sim \pi_{\theta}(\cdot \mid x)}\Big[ r_{\psi}(x, y) \Big]
\;-\; \beta\, \mathbb{E}_{x}\Big[ \mathrm{KL}\big[\pi_{\theta}(\cdot \mid x)\;\|\;\pi_{\text{ref}}(\cdot \mid x)\big] \Big]
\;+\; \gamma\, \mathbb{E}_{x \sim \mathcal{D}_{\text{pretrain}}}\big[ \log \pi_{\theta}(x) \big].
$$

- **Reward term**: what we want.
- **KL penalty** to the SFT model: the reward model was trained on samples *near*
  the SFT model and is only trustworthy there; the penalty keeps the policy in
  that region. It also keeps the text fluent, since the reward model never
  learned to penalize gibberish it never saw.
- **Pretraining term** (optional): a little next-token loss on pretraining data,
  to limit regression on benchmarks the reward model does not cover.


The first two terms are the **KL-regularized reward maximization** problem.
Section 4 solves it in closed form.


## Reward hacking and over-optimization

![Proxy reward keeps rising as the policy moves away from the reference; the quantity you wanted peaks and then falls. Illustrative curves using the functional form Gao et al. fit for RL. Source: instructor figure.](figs/14-RL-for-LLMs/kl-vs-reward.png)

[Gao et al. 2022](https://arxiv.org/abs/2210.10760) measured this with a large
"gold" reward model standing in for humans and smaller proxies trained on its
labels:

- the gold score as a function of $d = \sqrt{\mathrm{KL}}$ follows
  $d(\alpha - \beta \log d)$ for RL and $d(\alpha - \beta d)$ for best-of-$n$;
- larger reward models and more comparison data push the peak out;
- the KL penalty did not move the gold-vs-KL curve: at a given KL distance the
  gold score was the same with or without it, so the penalty acts like early
  stopping. It controls *how far* you go, not what happens when you get there.


Goodhart's law, quantified: **a learned reward is a proxy, and a strong
optimizer will find its errors.** Every RL run needs a measurement the policy
cannot optimize directly. The same curve appears in the wild: in DeepSeek-R1's
preference-rewarded final stage the learned reward kept rising while Codeforces
pass@1 fell, so the authors used it for only the last 400 of 1,700 steps and no
neural reward model at all for reasoning (section 5;
[DeepSeek-AI 2025](https://arxiv.org/abs/2501.12948), supplementary).


# DPO: skip the reward model

PPO needs sampling, a reward model, a value network and a KL penalty. Direct
Preference Optimization observes that the KL-regularized objective has a
closed-form solution, and that substituting it into the Bradley–Terry model
gives a loss on the *policy* that can be trained directly on the comparison
dataset, with no sampling at all. This is the lecture's second quizzable
derivation.


## The idea

- RLHF: comparisons $\to$ reward model $\to$ PPO $\to$ policy.
- The reward model is a stepping stone; what we want is the policy.
- [Rafailov et al. 2023](https://arxiv.org/abs/2305.18290): for the KL-regularized
  objective, **the optimal policy and the reward determine each other in closed
  form**. So write the reward *in terms of the policy*, plug it into the
  Bradley–Terry likelihood, and fit the policy to the comparisons directly.
- Result: a classification-style loss on $(x, y_w, y_l)$ triples. No sampling,
  no reward model, no value network. "Your language model is secretly a reward
  model."


## Step 1: the optimal policy under a KL penalty

Start from the RLHF objective without the pretraining term, for one prompt $x$:

$$
\max_{\pi}\; \mathbb{E}_{y \sim \pi(\cdot \mid x)}\big[ r(x, y) \big] - \beta\, \mathrm{KL}\big[\pi(\cdot \mid x)\,\|\,\pi_{\text{ref}}(\cdot \mid x)\big].
$$

Write out the KL, divide by $\beta$ and flip the sign to get a minimization:

$$
\min_{\pi}\; \mathbb{E}_{y \sim \pi}\!\left[ \log \frac{\pi(y \mid x)}{\pi_{\text{ref}}(y \mid x)} - \frac{1}{\beta} r(x, y) \right]
= \min_{\pi}\; \mathbb{E}_{y \sim \pi}\!\left[ \log \frac{\pi(y \mid x)}{\frac{1}{Z(x)} \pi_{\text{ref}}(y \mid x) \exp\!\big(r(x, y)/\beta\big)} - \log Z(x) \right],
$$

where we multiplied and divided inside the log by the normalizer
$Z(x) = \sum_{y} \pi_{\text{ref}}(y \mid x) \exp\!\big(r(x, y)/\beta\big)$, which
does not depend on $y$. The denominator is now a valid distribution,

$$
\pi^{*}(y \mid x) = \frac{1}{Z(x)}\, \pi_{\text{ref}}(y \mid x) \exp\!\Big( \frac{1}{\beta} r(x, y) \Big),
$$

so the objective is $\min_{\pi} \big( \mathrm{KL}[\pi \,\|\, \pi^{*}] - \log Z(x) \big)$. KL is
non-negative and zero only when the two distributions are equal, hence

$$
\boxed{\;\pi^{*}(y \mid x) = \frac{1}{Z(x)}\, \pi_{\text{ref}}(y \mid x)\, \exp\!\Big( \frac{1}{\beta} r(x, y) \Big)\;}
$$


## Step 2: invert, substitute, and watch $Z$ cancel

Take the log of the boxed equation and solve for the reward:

$$
r(x, y) = \beta \log \frac{\pi^{*}(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x).
$$

$Z(x)$ is intractable (a sum over all responses), but it is the same for every
$y$ given $x$. Substitute into the Bradley–Terry model, which only ever uses
*differences* of rewards for the same prompt:

$$
\mathrm{Pr}(y_w \succ y_l \mid x) = \sigma\!\left( \beta \log \frac{\pi^{*}(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi^{*}(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \right).
$$

The preference probability is now a function of the **policy alone**. Replace
$\pi^{*}$ by a parameterized $\pi_{\theta}$ and maximize the likelihood of the
comparison data:

$$
\boxed{\;L_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}}\!\left[ \log \sigma\!\left( \beta \log \frac{\pi_{\theta}(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_{\theta}(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \right) \right]\;}
$$

([Rafailov et al. 2023](https://arxiv.org/abs/2305.18290), eq. 7). Four
log-likelihoods per example, two forward passes through each of two models, and
an ordinary cross-entropy. $\beta$ is typically 0.1 to 0.5.


## What the DPO gradient does

Differentiating, with the implicit reward $\hat r_{\theta}(x, y) = \beta \log \frac{\pi_{\theta}(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$:

$$
\frac{\partial L_{\text{DPO}}}{\partial \theta}
= -\beta\, \mathbb{E}\Big[ \underbrace{\sigma\big(\hat r_{\theta}(x, y_l) - \hat r_{\theta}(x, y_w)\big)}_{\text{how wrong the implicit RM is}}
\Big( \underbrace{\frac{\partial \log \pi_{\theta}(y_w \mid x)}{\partial \theta}}_{\text{push } y_w \text{ up}}
- \underbrace{\frac{\partial \log \pi_{\theta}(y_l \mid x)}{\partial \theta}}_{\text{push } y_l \text{ down}} \Big) \Big].
$$

- Same shape as the policy gradient: log-probability of the good response up, of
  the bad one down, weighted by an advantage-like term.
- The weight is large when the model currently prefers the *wrong* response and
  shrinks to zero once it gets the pair right: a built-in focus on hard pairs.
- Without the weight you would get plain "SFT on $y_w$ minus SFT on $y_l$," which
  diverges; the derivation is what tells you the weight.


The loss and the gradient weight for three toy pairs, given each response's
log-ratio $\log \pi_{\theta}/\pi_{\text{ref}}$. A larger $\beta$ turns the same
log-ratios into a larger margin, so pairs the model already gets right are
switched off faster and wrong ones are pushed harder.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# (log pi_theta/pi_ref of the chosen y_w, same for the rejected y_l)
pairs = {"model already right": (+2.0, -2.0),
         "undecided":           ( 0.0,  0.0),
         "model wrong":         (-2.0, +2.0)}

for beta in (0.1, 0.5):
    print(f"beta = {beta}")
    for name, (logratio_w, logratio_l) in pairs.items():
        margin = beta * (logratio_w - logratio_l)   # r_hat(x, y_w) - r_hat(x, y_l)
        loss = -np.log(sigmoid(margin))
        weight = sigmoid(-margin)                   # sigma(r_hat(x, y_l) - r_hat(x, y_w))
        print(f"  {name:<20} margin {margin:+.2f}  loss {loss:.3f}  "
              f"weight {weight:.3f}  beta * weight {beta * weight:.3f}")

## DPO or PPO?

| | DPO | PPO / GRPO |
|---|---|---|
| data | a fixed set of $(x, y_w, y_l)$ | prompts only; responses are sampled |
| models in memory | 2 (policy, reference) | 3 to 4 |
| sampling during training | none | every step |
| reward | implicit in the comparisons | any program or model |
| off-policy risk | high: the pairs were written by *other* models; the policy drifts from them as it trains | low: data is always from the current policy |
| best for | style, safety, helpfulness: things people compare well | correctness, tool use, multi-step tasks: things a verifier scores |

In 2026 practice both are used in sequence: a DPO (or similar) stage on
preference data for chat behaviour, then an on-policy RL stage with verifiable
rewards for reasoning and agentic skills. Tülu 3 is a fully documented example
([Lambert et al. 2024](https://arxiv.org/abs/2411.15124)); the Qwen3 and
DeepSeek-R1 reports describe the same pattern.


# GRPO and verifiable rewards

PPO's value network and learned reward model were the expensive, fragile parts.
Group Relative Policy Optimization removes the value network by using *other
samples of the same prompt* as the baseline, and verifiable rewards remove the
reward model wherever a program can check the answer. Together they are the
recipe behind DeepSeek-R1-Zero and most 2025–26 reasoning models.


## GRPO: the group is the baseline

![One prompt, $G = 8$ sampled responses, a verifier's 0/1 reward, and the group-relative advantage. Three correct answers get $A = +1.29$, five wrong ones get $A = -0.77$. Source: instructor figure.](figs/14-RL-for-LLMs/grpo-group-sampling.png)

Check the arithmetic: with three of eight correct, the mean reward is $0.375$
and the (population) standard deviation is $\sqrt{0.375 \times 0.625} = 0.484$,
so a correct response gets $(1 - 0.375)/0.484 = +1.29$ and a wrong one
$(0 - 0.375)/0.484 = -0.77$ (population std; TRL and PyTorch use the sample
std, which gives $+1.21$/$-0.72$ for the same group). Note what happens if all eight are correct or all
eight are wrong: every advantage is zero and the prompt contributes nothing to
the gradient. That observation drives one of the fixes below.

The same arithmetic as a function, with both standard deviations and the
Dr. GRPO variant of section 5 (no std). The last lines show the difficulty
bias: GRPO multiplies the group-mean advantages by $1/\mathrm{std}$, which is
largest for groups that are nearly all right or all wrong.

In [ ]:
import numpy as np

def group_advantages(rewards, std="population"):
    r = np.asarray(rewards, dtype=float)
    centered = r - r.mean()
    if std is None:                                 # Dr. GRPO
        return centered
    s = r.std(ddof=0 if std == "population" else 1)
    return centered / (s + 1e-4)                    # the constant keeps all-same groups at 0

rewards = [1, 1, 1, 0, 0, 0, 0, 0]                  # 3 of G = 8 correct
r = np.array(rewards, dtype=float)
print(f"mean {r.mean():.3f}, population std {r.std():.3f}, sample std {r.std(ddof=1):.3f}")
for label, std in [("GRPO, population std", "population"),
                   ("GRPO, sample std (TRL)", "sample"),
                   ("Dr. GRPO, no std", None)]:
    A = group_advantages(rewards, std)
    print(f"{label:<23} correct {A[0]:+.2f}, wrong {A[-1]:+.2f}")
print("all 8 correct:", group_advantages([1] * 8))

for k in (1, 4, 7):
    group = [1] * k + [0] * (8 - k)
    print(f"{k} of 8 correct: GRPO scale 1/std = {1 / np.std(group):.2f}")

## The GRPO objective

[Shao et al. 2024](https://arxiv.org/abs/2402.03300) (DeepSeekMath), eq. 3; the
same objective is used for DeepSeek-R1. For each question $q$, sample a group
$\{o_1, \ldots, o_G\}$ from $\pi_{\theta_{\text{old}}}$, score each with reward
$r_i$, and maximize

$$
\mathcal{J}_{\text{GRPO}}(\theta) = \mathbb{E}\!\left[ \frac{1}{G} \sum_{i=1}^{G} \frac{1}{|o_i|} \sum_{t=1}^{|o_i|}
\Big( \min\!\big( \rho_{i,t}\, \hat A_{i,t},\; \mathrm{clip}(\rho_{i,t}, 1-\epsilon, 1+\epsilon)\, \hat A_{i,t} \big)
- \beta\, \mathbb{D}_{\text{KL}}\big[\pi_{\theta} \,\|\, \pi_{\text{ref}}\big] \Big) \right],
$$

with the token ratio $\rho_{i,t} = \pi_{\theta}(o_{i,t} \mid q, o_{i,<t}) / \pi_{\theta_{\text{old}}}(o_{i,t} \mid q, o_{i,<t})$ and the **group-relative advantage**, the same for every token of a response,

$$
\hat A_{i,t} = \frac{r_i - \mathrm{mean}(r_1, \ldots, r_G)}{\mathrm{std}(r_1, \ldots, r_G)}.
$$

The KL term is not the exact divergence but the per-token estimator

$$
\mathbb{D}_{\text{KL}}\big[\pi_{\theta} \,\|\, \pi_{\text{ref}}\big] = \frac{\pi_{\text{ref}}(o_{i,t} \mid q, o_{i,<t})}{\pi_{\theta}(o_{i,t} \mid q, o_{i,<t})} - \log \frac{\pi_{\text{ref}}(o_{i,t} \mid q, o_{i,<t})}{\pi_{\theta}(o_{i,t} \mid q, o_{i,<t})} - 1,
$$

which is unbiased and always non-negative ([Schulman 2020](http://joschu.net/blog/kl-approx.html)).

Check it on two distributions over five tokens, sampling from $\pi_{\theta}$:
the naive estimator $k_1 = \log(\pi_{\theta}/\pi_{\text{ref}})$ is also unbiased
but often negative, while $k_3$ never is and has lower variance.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
p_theta = np.array([0.50, 0.25, 0.15, 0.07, 0.03])
p_ref = np.array([0.40, 0.30, 0.10, 0.10, 0.10])
exact = np.sum(p_theta * np.log(p_theta / p_ref))

y = rng.choice(len(p_theta), size=100_000, p=p_theta)   # tokens sampled from pi_theta
ratio = p_ref[y] / p_theta[y]
k1 = -np.log(ratio)
k3 = ratio - np.log(ratio) - 1

print(f"exact KL[pi_theta || pi_ref] = {exact:.4f}")
for name, k in [("k1", k1), ("k3", k3)]:
    print(f"{name}: mean {k.mean():.4f}, std {k.std():.3f}, min {k.min():+.3f}, "
          f"negative in {np.mean(k < 0):.0%} of samples")

Compared with PPO: same clip, same ratio; the advantage comes from the group
instead of a value network; the KL is a loss term instead of a per-token reward.


## Why no critic

- **Memory and compute.** The value network is a second full-size model with its
  own optimizer state. Dropping it roughly halves the training footprint.
- **The value target is ill-posed for long reasoning.** The critic must predict
  the final reward from a partial response; with reflection and revision, the
  first half of a response is often contradicted by the second half. The R1
  report gives this as the reason, not just the cost.
- **The group baseline is the right baseline for a terminal reward.** With one
  reward per response, the best action-independent baseline for prompt $q$ is the
  expected reward of the current policy on $q$; the group mean is a Monte Carlo
  estimate of exactly that. It is unbiased by the argument of section 2.
- **Cost.** $G$ generations per prompt instead of one. Generation was already the
  bottleneck, so this is where the step time goes.


The R1 report's ablation: PPO with default $\lambda = 0.95$ did considerably
worse than GRPO on MATH; carefully tuned ($\lambda = 1$) it came close
([DeepSeek-AI 2025](https://arxiv.org/abs/2501.12948), supplementary section
on GRPO vs. PPO). GRPO's advantage is robustness, not a fundamentally better
estimator.


## RLVR: rewards you can verify

**Reinforcement learning with verifiable rewards** (the name is from Tülu 3,
[Lambert et al. 2024](https://arxiv.org/abs/2411.15124)): wherever a program can
decide whether a response is correct, use it instead of a reward model.

**DeepSeek-R1-Zero's reward** ([DeepSeek-AI 2025](https://arxiv.org/abs/2501.12948)):

- **Accuracy reward.** Math: the final answer, required in a fixed format
  (e.g. `\boxed{}`), matches the reference. Code: the program passes the test
  cases.
- **Format reward.** The reasoning is enclosed in `<think>` ... `</think>` tags.
- The two are combined (the paper does not state the weights). **No neural
  reward model at all** for reasoning data.

**Why this changes everything about section 3:**

- No reward model to over-optimize (though graders can still be gamed, section 7).
- No limit from the labeler: the model can exceed the best human-written
  solution if the checker says so.
- Rewards are cheap and parallel; you can afford $G = 16$ or $64$ samples and
  thousands of steps.
- Only works where a checker exists: math with a reference answer, code with
  tests, logic puzzles, structured extraction, anything with a unit test.


## DeepSeek-R1-Zero: RL on a base model, no SFT

![AIME 2024 accuracy of DeepSeek-R1-Zero during RL training: pass@1 and majority vote over 16 samples, against the average score of human AIME participants. Figure 1a of [DeepSeek-AI 2025](https://arxiv.org/abs/2501.12948), arXiv v2 (the *Nature* version).](figs/14-RL-for-LLMs/r1-zero-aime-during-training.png)

Start from DeepSeek-V3-Base (671B total, 37B active), skip SFT, run GRPO with
the accuracy and format rewards above on reasoning prompts, for about ten
thousand policy updates. The paper reports AIME 2024 pass@1 rising from 15.6%
to 77.9% over training (the January 2025 preprint reported 71.0% for an earlier
checkpoint, with 86.7% by majority vote over 64 samples). Those are the paper's
numbers; the lecture quotes nothing else. What the authors found remarkable
was not the number but *how* it was reached.


## What the model learned to do: think longer

![Average response length of DeepSeek-R1-Zero on the training set during RL. Figure 1b of [DeepSeek-AI 2025](https://arxiv.org/abs/2501.12948).](figs/14-RL-for-LLMs/r1-zero-response-length.png)

Nobody rewarded length. The reward is 0 or 1 on the final answer. Yet the
policy found that spending more tokens on re-checking, trying alternatives and
backtracking raises the probability of a correct final answer, and the gradient
pushed it there.

The report documents an "aha moment" in an intermediate checkpoint: the model
writes *"Wait, wait. Wait. That's an aha moment I can flag here"* and restarts a
derivation, and the frequency of the word "wait" jumps during training.


## Two honest caveats

- **The base model already had it.** [Liu et al. 2025](https://arxiv.org/abs/2503.20783)
  show that DeepSeek-V3-Base *already* produces "aha"-style reflection before
  any RL, and that Qwen2.5 base models reason well with no template at all. RL
  amplifies behaviours the base model has; the base checkpoint matters as much
  as the algorithm. The R1 report itself notes that 7B and 16B base models
  failed to improve on AIME under the same recipe.
- **Sharpening, not (yet) discovery.** [Yue et al. 2025](https://arxiv.org/abs/2504.13837)
  find that at large $k$ the *base* model's pass@$k$ matches or exceeds the RL
  model's: current RLVR concentrates probability on solutions the base model
  could already sample, rather than creating new ones. Distillation from a
  stronger teacher, by contrast, does add new reasoning patterns.


Neither caveat makes the result less useful: pass@1 is what a user experiences.
They do change what you should *expect* from an RL run on a given base, and how
to evaluate it (section 7).


## Pitfall 1: length bias, and Dr. GRPO

[Liu et al. 2025](https://arxiv.org/abs/2503.20783) ("Understanding R1-Zero-like
training") point at two normalizations in the GRPO objective:

- **The $1/|o_i|$ length normalization.** A *wrong* response (negative advantage)
  that is long has its per-token penalty divided by a large $|o_i|$, so being
  long is a way to be punished less; a *correct* response that is short gets a
  larger per-token reward. The net effect is that incorrect responses grow
  longer over training: length growth that is an optimizer artifact, not
  "thinking."
- **The $1/\mathrm{std}$ reward normalization.** Prompts the model nearly always
  gets right or nearly always gets wrong have small std and are upweighted; a
  difficulty bias in which questions dominate the gradient.


**Dr. GRPO** ("GRPO done right") deletes both: sum the per-token terms without
dividing by $|o_i|$ (equivalently divide by a constant such as the maximum
length), and use $\hat A_i = r_i - \mathrm{mean}(r)$ with no std. This recovers
the plain policy gradient with a group-mean baseline. Their minimalist recipe
reaches 43.3% on AIME 2024 with a 7B base model, with shorter responses than
GRPO. In TRL this is `loss_type="dr_grpo"` and `scale_rewards=False`.


## Pitfall 2: entropy collapse and empty groups: DAPO

[Yu et al. 2025](https://arxiv.org/abs/2503.14476) (DAPO, ByteDance Seed and
Tsinghua) trained Qwen2.5-32B-Base to 50 points on AIME 2024 and published the
four changes that made it work:

- **Clip-higher.** With a symmetric clip, a token at probability 0.01 can grow to
  at most 0.012 per update while one at 0.9 can grow to 1.0. Exploration tokens
  are capped; the policy's entropy collapses early. DAPO decouples the range:
  $\epsilon_{\text{low}} = 0.2$, $\epsilon_{\text{high}} = 0.28$. In TRL:
  `epsilon_high`.
- **Dynamic sampling.** A group with all-correct or all-wrong answers has zero
  advantage everywhere and contributes no gradient; as the model improves, more
  groups are like that and the effective batch shrinks. DAPO over-samples and
  keeps only prompts with mixed outcomes, so every batch has a full gradient.
- **Token-level loss.** Average over all tokens in the batch rather than per
  response then per group, so long responses are neither under- nor
  over-weighted. Same concern as Dr. GRPO, a different normalizer
  (`loss_type="dapo"`).
- **Overlong reward shaping.** A truncated response gets a soft, length-dependent
  penalty rather than a hard 0, which was adding noise to the reward.


What each fix addresses: **collapse of exploration** (clip-higher), **wasted
compute and vanishing signal** (dynamic sampling), **length bias** (token-level
loss), and **reward noise from truncation** (overlong shaping).


## The 2025–26 recipe, honestly

| Choice | GRPO (2024) | Common practice 2025–26 | Addresses |
|---|---|---|---|
| advantage | $(r_i - \text{mean})/\text{std}$ | group mean, often no std | difficulty bias |
| loss normalization | per response, then per group | token-level or constant | length bias |
| clip | symmetric $\epsilon = 0.2$ | $\epsilon_{\text{high}} > \epsilon_{\text{low}}$ | entropy collapse |
| sampling | fixed $G$ | drop all-same groups, resample | wasted gradient |
| ratio | per token | per token, or per sequence (GSPO) | noise, MoE instability |
| KL to reference | $\beta > 0$ | often $\beta = 0$ ([Hu et al. 2025](https://arxiv.org/abs/2503.24290), DAPO, Dr. GRPO) | cost of a 3rd model; the clip already limits the step |
| reference | fixed SFT model | reset periodically (R1), or none | long runs drift far from the start |

The GSPO row: [Zheng et al. 2025](https://arxiv.org/abs/2507.18071) (the Qwen
team) clip one length-normalized *sequence-level* ratio instead of thousands of
noisy token ratios, which matches the granularity of the reward and was what
made RL on mixture-of-experts models (lecture 11), where routing changes make
token ratios swing, converge for Qwen3 (`importance_sampling_level="sequence"`
in TRL).

Two things to keep straight. (1) None of these changes the *estimator* in
section 2; they change the baseline, the normalizer and the trust region.
(2) The base model matters as much as the algorithm: [Shao et al. 2025](https://arxiv.org/abs/2506.10947)
got a 21-point MATH-500 gain on Qwen2.5-Math-7B with *random* rewards (the clip
amplifies behaviours the base already favours), and the same recipe does
nothing for Llama 3 or OLMo 2. Always run the random-reward control.


# Agentic RL

Everything so far was one prompt, one response, one reward. The frontier in
2025–26 is RL over *multi-turn* trajectories in which the model calls tools,
reads their output and continues. The algorithms are the same; the rollout and
the reward are different. Lectures 21 and 22 take this up in depth; here is the
one slide that connects them to the math.


## Multi-turn rollouts with tool calls

![One agentic trajectory. The policy's tokens (orange) are actions and receive gradient; tool outputs (gray) are observations and are masked out of the loss; one sparse reward arrives at the end. Source: instructor figure.](figs/14-RL-for-LLMs/agentic-rollout.png)

- **State** is the whole transcript; **actions** are only the policy's tokens.
  Tool outputs enter the context but get no gradient (they are the environment's
  transition, which section 2 showed drops out).
- **Reward**: did the tests pass, was the task completed, did a judge accept
  the final state. Sparse and terminal, like R1-Zero, but after many more tokens
  and real side effects.
- The group-relative advantage still works: sample $G$ trajectories for the same
  task, compare outcomes.


## Agentic RL in practice is lecture 22's topic

The estimator does not change; the rollout infrastructure does. Lecture 21
covers the agent loop and harness these trajectories run in; lecture 22 covers
asynchronous rollouts (trajectories differ in length by orders of magnitude, so
a trainer consumes finished ones from a queue fed by sandboxed environments),
tool protocols, agent evaluation, and where this kind of training fails.

# Practicalities on one GPU

The R1-Zero run used a 671B model and thousands of GPUs. The same algorithm on a
small model and a small task fits on one SCC GPU in an afternoon, and that is
the point of this section: what to run, what it costs, and how to tell whether
it worked.


## What you can actually run

**Software.** TRL's `GRPOTrainer`
([docs](https://huggingface.co/docs/trl/grpo_trainer)): pass a model name, a
prompt dataset and one or more reward *functions*; it samples, scores, computes
the group advantage and trains. Pass a `peft_config` (a `LoraConfig`) and it
trains LoRA adapters (lecture 13); the reference model is then just the base
weights with the adapter switched off, so there is no second copy. `use_vllm=True` moves
generation to vLLM. The loss variants of section 5 are configuration flags.

**Model.** A 1–4B open-weight *base* checkpoint (for example Qwen3-1.7B-Base or
the current small Qwen or Gemma base model). TinyZero found that a 0.5B model
failed to learn and 1.5B to 3B models did.

**Task: Countdown** (the TinyZero reproduction of R1-Zero,
[Pan et al. 2025](https://github.com/Jiayi-Pan/TinyZero)). Given numbers such as
$\{19, 36, 55, 7\}$ and a target such as $65$, write an arithmetic expression
using each number once that evaluates to the target.

```python
import re

def countdown_reward(completions, numbers, target, **kw):
    scores = []
    for text, nums, tgt in zip(completions, numbers, target):
        m = re.search(r"<answer>(.*?)</answer>", text, re.S)
        if m is None:                 # no answer tag
            scores.append(0.0)
            continue
        expr = m.group(1).strip()
        used = sorted(map(int, re.findall(r"\d+", expr)))
        try:
            # never eval() untrusted output in production; use a safe arithmetic parser
            ok = used == sorted(nums) and \
                 abs(eval(expr) - tgt) < 1e-6
        except Exception:
            ok = False
        scores.append(1.0 if ok else 0.1)  # format only
    return scores
```

The checker is forty lines; the model is the only thing that learns. TinyZero
reports the 3B model developing self-verification and search behaviour for
under \$30 of compute.


## Compute per step, by the rules of lecture 12

Take a 1.7B model, 16 prompts $\times$ $G = 8$ completions, up to 1,024 new tokens
each: about $1.3 \times 10^5$ generated tokens per step.

- **Training FLOPs.** $6ND$ with $N = 1.7 \times 10^9$, $D = 1.3 \times 10^5$:
  about $1.3 \times 10^{15}$ FLOPs, about 13 s at the 100 TFLOP/s one A100
  reaches in practice. With LoRA the frozen weights skip their weight-gradient
  matmuls, so a step is closer to $4ND \approx 9$ s.
- **Reference and old-policy log-probabilities.** Two extra forward passes over
  the same tokens: about 4.5 s each ($2ND \approx 4.5 \times 10^{14}$ FLOPs).
- **Generation.** 1,024 sequential decode steps. Each step reads all 3.4 GB of
  bf16 weights plus the KV cache (about 0.11 MB per token with GQA, so up to
  about 15 GB at the end): memory-bound, roughly 2 to 9 ms per step on an A100
  depending on batch and cache size, so **several seconds to tens of seconds per
  step for generation**, and far more with plain `generate()` than with vLLM.
- **Memory.** Policy 3.4 GB; LoRA optimizer state negligible; KV cache up to
  about 13 GB; activations for the training pass on 128 sequences $\times$ 1,024
  tokens, manageable with gradient checkpointing. Fits a 40 GB GPU.


So with vLLM the step splits roughly evenly between generation and the three
passes over the tokens, and with plain `generate()` it is **generation-bound**,
as the arithmetic predicted in section 2. A few hundred steps is an afternoon
on one GPU, which is enough to watch the
reward curve on Countdown go from format-only to mostly correct.


The FLOP arithmetic above as a calculator; change the model size, group size or
response length and see which term grows.

In [ ]:
N = 1.7e9                  # parameters
prompts, G = 16, 8         # prompts per step, completions per prompt
max_new_tokens = 1024
throughput = 100e12        # FLOP/s one A100 reaches in practice
bytes_per_param = 2        # bf16

D = prompts * G * max_new_tokens
print(f"generated tokens per step: D = {D:,} (about {D:.1e})")
print(f"policy weights: {N * bytes_per_param / 1e9:.1f} GB")
for name, flops_per_token in [("training, 6N", 6 * N),
                              ("training with LoRA, ~4N", 4 * N),
                              ("one log-prob pass, 2N", 2 * N)]:
    flops = flops_per_token * D
    print(f"{name:<24} {flops:.2e} FLOPs = {flops / throughput:5.1f} s")

## Did the reward go up, or did the model learn the grader?

A rising training reward is necessary, not sufficient. Before believing a run:

- **Held-out accuracy**, on prompts never used for RL, scored by the same
  verifier. Report pass@1 averaged over several samples, as the R1 report does
  ($k = 16$ to $64$ samples at temperature 0.6 to 0.7).
- **Read the samples.** A format reward can be satisfied without an answer; an
  answer extractor can be fooled by printing several candidates; `eval` on a
  string can be gamed by an expression that is not arithmetic. The R1 authors
  put rule-based reward *design* first for this reason.
- **The random-reward control.** Train the same model with rewards drawn at
  random. If it improves nearly as much
  ([Shao et al. 2025](https://arxiv.org/abs/2506.10947)), your gain is coming
  from the base model and the clip, not from your reward.
- **pass@$k$ at large $k$** against the base model ([Yue et al. 2025](https://arxiv.org/abs/2504.13837)):
  if the base model with 64 samples matches your RL model with 1, you sharpened
  rather than taught.
- **Three curves to log every step:** mean reward, mean response length, and
  policy entropy (or KL to the reference). Length rising with flat accuracy is
  pitfall 1; entropy falling to zero is pitfall 2; a reward that rises while
  held-out accuracy falls is section 3's over-optimization.


# Summary

One line per idea; the recap deck uses these.


## Summary

- SFT is maximum likelihood on demonstrations: no negative signal, no program-computed rewards, never sees the model's own samples. RL supplies all three at the cost of sampling.
- An LLM is a policy: state = prefix, action = next token, trajectory = response, one terminal reward.
- Policy-gradient theorem: $\partial J / \partial \boldsymbol{\theta} = \mathbb{E}_{\tau}[\sum_t \partial \log \pi[a_t \mid s_t, \boldsymbol{\theta}] / \partial \boldsymbol{\theta}\; r[\tau]]$; the transition terms vanish, so no environment model is needed.
- A baseline $b$ independent of the action leaves the expectation unchanged ($\mathbb{E}[\partial \log \pi / \partial \boldsymbol{\theta}] = 0$) and cuts variance; the choice of baseline is the main difference between PPO (value network) and GRPO (group mean).
- Importance ratio $\rho_t = \pi_{\theta}/\pi_{\theta_{\text{old}}}$ lets one batch serve several updates; PPO clips it to $[1-\epsilon, 1+\epsilon]$ and takes the pessimistic $\min$.
- Reward models fit Bradley–Terry, $\mathrm{Pr}(y_w \succ y_l) = \sigma(r(x, y_w) - r(x, y_l))$, so rewards are defined up to a per-prompt constant.
- RLHF maximizes $\mathbb{E}[r] - \beta\,\mathrm{KL}[\pi_{\theta} \| \pi_{\text{ref}}]$; a learned reward is a proxy and the gold score peaks then falls with KL (over-optimization).
- The KL-regularized optimum is $\pi^{*} \propto \pi_{\text{ref}} \exp(r/\beta)$; inverting and substituting into Bradley–Terry cancels $Z(x)$ and gives the DPO loss, a classification loss on comparison pairs with no sampling.
- GRPO: $G$ samples per prompt, advantage $(r_i - \text{mean})/\text{std}$, same clip as PPO, KL as a loss term, no critic.
- RLVR: rule-based accuracy and format rewards wherever a checker exists; R1-Zero reached 77.9% AIME 2024 pass@1 from a base model with no SFT, and its responses grew to over 10k tokens unprompted.
- Fixes and what they address: Dr. GRPO (length and difficulty bias from the normalizers), DAPO (clip-higher for entropy collapse, dynamic sampling for empty groups, token-level loss, overlong shaping), GSPO (sequence-level ratio for noise and MoE instability); $\beta = 0$ is now common.
- Agentic RL: same estimator over multi-turn trajectories; tool outputs are masked observations; rewards are task outcomes; the rollout infrastructure is lecture 22's topic.
- On one GPU: TRL `GRPOTrainer` with LoRA on a 1–4B base model and a verifiable task; the step is generation-bound; validate with held-out accuracy, a random-reward control, pass@$k$ against the base, and by reading samples.

## Further reading

- UDL chapter 19, sections 19.1 and 19.5: the MDP vocabulary and the policy-gradient derivation this lecture follows.
- [Schulman et al., *Proximal Policy Optimization Algorithms* (2017)](https://arxiv.org/abs/1707.06347) and [Schulman, *Approximating KL divergence* (2020)](http://joschu.net/blog/kl-approx.html): the clip and the $k_3$ estimator.
- [Ouyang et al., *Training language models to follow instructions with human feedback* (2022)](https://arxiv.org/abs/2203.02155): RLHF as run for InstructGPT.
- [Rafailov et al., *Direct Preference Optimization* (2023)](https://arxiv.org/abs/2305.18290): the derivation of section 4, with the proof in appendix A.
- DPO variants, by name only: [IPO](https://arxiv.org/abs/2310.12036) (squared loss on the margin), [KTO](https://arxiv.org/abs/2402.01306) (single-response thumbs-up/down labels), [ORPO](https://arxiv.org/abs/2403.07691) (no reference model), [SimPO](https://arxiv.org/abs/2405.14734) (length-normalized implicit reward). If you can derive DPO you can read all four in an afternoon.
- [Shao et al., *DeepSeekMath* (2024)](https://arxiv.org/abs/2402.03300) and [DeepSeek-AI, *DeepSeek-R1* (2025)](https://arxiv.org/abs/2501.12948): GRPO, R1-Zero, and the supplementary comparison with PPO.
- [Liu et al., *Understanding R1-Zero-Like Training* (2025)](https://arxiv.org/abs/2503.20783), [Yu et al., *DAPO* (2025)](https://arxiv.org/abs/2503.14476), [Zheng et al., *GSPO* (2025)](https://arxiv.org/abs/2507.18071): the fixes of section 5, each with ablations.
- [Lambert et al., *Tülu 3* (2024)](https://arxiv.org/abs/2411.15124): a fully open post-training recipe with SFT, DPO and RLVR stages.
- [TRL GRPO trainer documentation](https://huggingface.co/docs/trl/grpo_trainer): every loss variant above as a configuration flag, with the paper each one comes from.